## 설비 고장 예측 - Baseline

**접근법 (main_1 기반)**
- 전체 정상 데이터(`Target == 0`)를 오토인코더 학습에 사용
- `HIDDEN_SIZE=20`, 15 epochs
- Stage 2 분류: 고장 행 직접 추출 후 슬라이딩 윈도우

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import torch
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from src.config import *
from src.models import LSTMAutoencoder, LSTMClassifier
from src.data_utils import load_and_engineer, create_windows, prepare_classifier_data_simple
from src.train_utils import train_autoencoder, compute_errors
from src.visualization import plot_confusion_matrix

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'디바이스: {device}')

### 1. 데이터 전처리

In [ ]:
df = load_and_engineer(DATA_PATH)

# 정상 데이터 전체를 오토인코더 학습에 사용
train_df = df[df['Target'] == 0]
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_df[FEATURES])
test_scaled = scaler.transform(df[FEATURES])

X_train, y_train = create_windows(train_scaled, SEQUENCE_LENGTH)
X_test, y_test = create_windows(test_scaled, SEQUENCE_LENGTH)

train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
test_dataset = TensorDataset(torch.FloatTensor(X_test), torch.FloatTensor(y_test))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
train_eval_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'훈련 시퀀스: {X_train.shape}, 테스트 시퀀스: {X_test.shape}')

### 2. 오토인코더 학습 (Stage 1)

In [ ]:
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

model = LSTMAutoencoder(INPUT_SIZE, BASELINE_HIDDEN_SIZE, SEQUENCE_LENGTH).to(device)
history = train_autoencoder(
    model, train_loader, test_loader,
    BASELINE_EPOCHS, LR, device,
    f'{CHECKPOINT_DIR}/autoencoder_v1.pth'
)

### 3. 이상 탐지 평가 (Stage 1)

In [ ]:
model.load_state_dict(torch.load(f'{CHECKPOINT_DIR}/autoencoder_v1.pth'))

train_errors = compute_errors(model, train_eval_loader, device)
test_errors = compute_errors(model, test_loader, device)

# 임계값 비교 (95 / 90 / 80 퍼센타일)
y_true = df['Target'].values[SEQUENCE_LENGTH - 1:].astype(int)
for p in [95, 90, 80]:
    threshold = np.percentile(train_errors, p)
    y_pred = (test_errors > threshold).astype(int)
    print(f'[{p}th percentile | threshold: {threshold:.6f}]')
    print(classification_report(y_true, y_pred, target_names=['Normal', 'Anomaly'], zero_division=0))

In [ ]:
# 95th percentile 기준 혼동 행렬
threshold = np.percentile(train_errors, 95)
y_pred = (test_errors > threshold).astype(int)
plot_confusion_matrix(y_true, y_pred, ['Normal', 'Anomaly'], '이상 탐지 혼동 행렬 (Stage 1)')

### 4. 고장 유형 분류 (Stage 2)

In [ ]:
failure_df = df[df['Target'] == 1].copy()
le = LabelEncoder()
failure_df['label'] = le.fit_transform(failure_df['Failure Type'])
print('클래스:', le.classes_)

failure_scaled = scaler.transform(failure_df[FEATURES])
failure_labels = failure_df['label'].values

X_fail, y_fail = prepare_classifier_data_simple(failure_scaled, failure_labels, SEQUENCE_LENGTH)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_fail, y_fail, test_size=0.2, random_state=42, stratify=y_fail
)

num_classes = len(le.classes_)
clf_train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr)), batch_size=16, shuffle=True
)
clf_test_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_te), torch.LongTensor(y_te)), batch_size=16, shuffle=False
)

classifier = LSTMClassifier(INPUT_SIZE, 32, num_classes).to(device)
clf_criterion = torch.nn.CrossEntropyLoss()
clf_optimizer = torch.optim.Adam(classifier.parameters(), lr=LR)

for epoch in range(1, 51):
    classifier.train()
    for inputs, labels in clf_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        clf_optimizer.zero_grad()
        clf_criterion(classifier(inputs), labels).backward()
        clf_optimizer.step()

classifier.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in clf_test_loader:
        _, predicted = torch.max(classifier(inputs.to(device)), 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=le.classes_, zero_division=0))
plot_confusion_matrix(all_labels, all_preds, le.classes_, '고장 유형 분류 혼동 행렬 (Stage 2)')